# 02. Validación de unidades textuales — ONPE 2026

## Objetivo
Comprobar si los archivos de texto generados a partir de los JSON de ONPE se crearon correctamente y conservan información necesaria para recuperar evidencia: referencia a la fuente, ruta de origen, campos, valores y contexto geográfico disponible.

Ejecuta primero el script `generar_unidades.py` si todavía no existen unidades textuales.

Rutas esperadas:
- JSON originales: `data/raw/ONPE_2026/Peru/`
- Unidades textuales: `data/processed/unidades_textuales/`
- Notebook: `notebooks/02_validacion_unidades_textuales.ipynb`


## 1. Preparar el entorno y localizar las carpetas

La búsqueda de la raíz intenta encontrar las carpetas `data/raw` y `data/processed`. 

In [ ]:
from pathlib import Path
import re
import json
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_columns", 60)

def encontrar_raiz(inicio: Path) -> Path:
    inicio = inicio.resolve()
    for candidato in [inicio, *inicio.parents]:
        if (candidato / "data" / "raw").exists() or (candidato / "data" / "processed").exists():
            return candidato
        if (candidato / "README.md").exists() and (candidato / "notebooks").exists():
            return candidato
    return inicio.parent if inicio.name == "notebooks" else inicio

RAIZ = encontrar_raiz(Path.cwd())
RUTA_JSON = RAIZ / "data" / "raw" / "ONPE_2026" / "Peru"
RUTA_TEXTOS = RAIZ / "data" / "processed" / "unidades_textuales"

print("Directorio actual:", Path.cwd().resolve())
print("Raíz detectada:", RAIZ)
print("JSON originales:", RUTA_JSON, "| existe:", RUTA_JSON.exists())
print("Unidades textuales:", RUTA_TEXTOS, "| existe:", RUTA_TEXTOS.exists())


## 2. Inventario de unidades textuales

Contamos los `.txt`, su tamaño, número de líneas y ubicación relativa. 

In [ ]:
archivos_txt = sorted(RUTA_TEXTOS.rglob("*.txt")) if RUTA_TEXTOS.exists() else []
filas = []
for archivo in archivos_txt:
    try:
        contenido = archivo.read_text(encoding="utf-8-sig")
        rel = archivo.relative_to(RUTA_TEXTOS)
        filas.append({
            "archivo": archivo.name,
            "ruta_relativa": rel.as_posix(),
            "tamano_bytes": archivo.stat().st_size,
            "cantidad_lineas": len(contenido.splitlines()),
            "vacio_o_solo_espacios": not contenido.strip(),
        })
    except Exception as exc:
        filas.append({"archivo": archivo.name, "ruta_relativa": str(archivo),
                      "tamano_bytes": archivo.stat().st_size, "cantidad_lineas": None,
                      "vacio_o_solo_espacios": None, "error_lectura": str(exc)[:200]})

df_textos = pd.DataFrame(filas)
print(f"Unidades textuales encontradas: {len(df_textos):,}")
if not df_textos.empty:
    display(df_textos.head(20))
    print("Tamaño total (MB):", round(df_textos["tamano_bytes"].sum() / 1024**2, 2))
else:
    print("No hay unidades textuales. Ejecuta el generador y vuelve a ejecutar esta sección.")


## 3. Comprobar metadatos de fuente y archivo de origen

Una unidad debe permitir rastrear de qué JSON procede. Los generadores del proyecto pueden escribir `FUENTE:` y/o `ARCHIVO_ORIGEN:`.La ruta de origen se usa para comprobar que el JSON original existe.

In [ ]:
validaciones = []
for archivo in archivos_txt:
    try:
        texto = archivo.read_text(encoding="utf-8-sig")
        lineas = texto.splitlines()
        fuente = next((x.split(":", 1)[1].strip() for x in lineas if x.startswith("FUENTE:")), "")
        origen = next((x.split(":", 1)[1].strip() for x in lineas if x.startswith("ARCHIVO_ORIGEN:")), "")
        # La ruta relativa se prefiere cuando está disponible; también se admite una ruta absoluta.
        candidato_origen = None
        if origen:
            candidato = Path(origen)
            candidato_origen = candidato if candidato.is_absolute() else RUTA_JSON / candidato
        elif fuente:
            candidato = Path(fuente)
            if candidato.is_absolute() and candidato.exists():
                candidato_origen = candidato
            else:
                # Algunos generadores antiguos escribían la ruta relativa desde la raíz del proyecto.
                opciones = [RAIZ / candidato, RUTA_JSON / candidato]
                candidato_origen = next((p for p in opciones if p.exists()), opciones[0])
        existe_origen = candidato_origen.exists() if candidato_origen is not None else False
        validaciones.append({
            "unidad": archivo.relative_to(RUTA_TEXTOS).as_posix(),
            "tiene_marcador_fuente": bool(fuente),
            "tiene_archivo_origen": bool(origen),
            "tiene_metadato_rastreo": bool(fuente or origen),
            "ruta_origen_resuelta": str(candidato_origen) if candidato_origen else "",
            "json_origen_existe": existe_origen,
            "tiene_contenido": bool(texto.strip()),
            "lineas_con_contenido": sum(bool(x.strip()) for x in lineas),
        })
    except Exception as exc:
        validaciones.append({"unidad": str(archivo), "tiene_marcador_fuente": False,
                             "tiene_archivo_origen": False, "tiene_metadato_rastreo": False,
                             "ruta_origen_resuelta": "", "json_origen_existe": False,
                             "tiene_contenido": False, "lineas_con_contenido": 0,
                             "error": str(exc)[:200]})

df_rastreo = pd.DataFrame(validaciones)
if not df_rastreo.empty:
    resumen_rastreo = pd.DataFrame({
        "control": ["Tiene marcador FUENTE", "Tiene ARCHIVO_ORIGEN", "Tiene algún metadato de rastreo",
                    "El JSON de origen existe", "Tiene contenido no vacío"],
        "cumplen": [int(df_rastreo["tiene_marcador_fuente"].sum()),
                    int(df_rastreo["tiene_archivo_origen"].sum()),
                    int(df_rastreo["tiene_metadato_rastreo"].sum()),
                    int(df_rastreo["json_origen_existe"].sum()),
                    int(df_rastreo["tiene_contenido"].sum())],
        "total": [len(df_rastreo)] * 5,
    })
    resumen_rastreo["porcentaje"] = (resumen_rastreo["cumplen"] / resumen_rastreo["total"] * 100).round(2)
    display(resumen_rastreo)
    display(df_rastreo[(~df_rastreo["tiene_metadato_rastreo"]) | (~df_rastreo["json_origen_existe"]) | (~df_rastreo["tiene_contenido"])].head(50))


## 4. Comparar cantidad de JSON y unidades textuales

En principio, el generador produce una unidad `.txt` por cada JSON procesado. La diferencia ayuda a localizar archivos no procesados o salidas faltantes.

In [ ]:
archivos_json = sorted(RUTA_JSON.rglob("*.json")) if RUTA_JSON.exists() else []

def normalizar_ruta_relativa(ruta):
    partes = list(Path(ruta).parts)
    # Algunas versiones del generador conservan "Peru/" en la salida,
    # aunque RUTA_JSON ya apunta al directorio .../ONPE_2026/Peru.
    if partes and partes[0].lower() == "peru":
        partes = partes[1:]
    return Path(*partes).as_posix() if partes else ""

rel_json = {
    normalizar_ruta_relativa(
        p.relative_to(RUTA_JSON).with_name(p.stem + "_unidad_textual.txt")
    )
    for p in archivos_json
}
rel_txt = {
    normalizar_ruta_relativa(p.relative_to(RUTA_TEXTOS))
    for p in archivos_txt
} if RUTA_TEXTOS.exists() else set()

faltantes = sorted(rel_json - rel_txt)
extras = sorted(rel_txt - rel_json)
comparacion = pd.DataFrame([
    {"indicador": "JSON originales", "cantidad": len(archivos_json)},
    {"indicador": "Unidades textuales", "cantidad": len(archivos_txt)},
    {"indicador": "Unidades esperadas que faltan", "cantidad": len(faltantes)},
    {"indicador": "Unidades sin JSON correspondiente por nombre/ruta", "cantidad": len(extras)},
])
display(comparacion)
if faltantes:
    print("Primeras unidades esperadas faltantes:")
    display(pd.DataFrame({"unidad_faltante": faltantes[:50]}))
if extras:
    print("Primeras unidades adicionales o con ruta/nombre distinto:")
    display(pd.DataFrame({"unidad_adicional": extras[:50]}))


## 5. Inspeccionar una unidad textual de ejemplo

Se muestra una unidad representativa para verificar visualmente si conserva el nombre del campo y su valor. No se imprimen todas las unidades

In [ ]:
if archivos_txt:
    # Preferimos un resumen de participantes para ver campos de candidato y votos.
    candidatas = [p for p in archivos_txt if "participantes" in p.name.lower()]
    ejemplo = candidatas[0] if candidatas else archivos_txt[0]
    texto_ejemplo = ejemplo.read_text(encoding="utf-8-sig")
    print("Unidad seleccionada:", ejemplo.relative_to(RUTA_TEXTOS))
    print("Longitud en caracteres:", len(texto_ejemplo))
    print("-" * 90)
    print("\n".join(texto_ejemplo.splitlines()[:100]))
else:
    print("No hay unidades para mostrar. Genera los textos antes de continuar.")


## 6. Verificar conservación de campos relevantes

Buscamos nombres de campos que deberían aparecer en las unidades derivadas de resúmenes y participantes. La validación se hace por tipo de archivo: un campo puede no aplicar a todos los JSON.

In [ ]:
campos_por_tipo = {
    "participantes": ["nombreAgrupacionPolitica", "nombreCandidato", "totalVotosValidos",
                      "porcentajeVotosValidos", "porcentajeVotosEmitidos"],
    "totales": ["totalVotosEmitidos", "totalVotosValidos", "participacionCiudadana",
                "actasContabilizadas", "totalActas"],
    "catalogos": ["nombre", "ubigeo"],
}
chequeos = []
for archivo in archivos_txt:
    nombre = archivo.name.lower()
    if "participantes" in nombre:
        tipo = "participantes"
    elif "totales" in nombre:
        tipo = "totales"
    elif any(x in nombre for x in ["departamentos", "provincias", "distritos"]):
        tipo = "catalogos"
    else:
        tipo = "otro"
    if tipo == "otro":
        continue
    texto = archivo.read_text(encoding="utf-8-sig")
    for campo in campos_por_tipo[tipo]:
        chequeos.append({"unidad": archivo.relative_to(RUTA_TEXTOS).as_posix(),
                         "tipo": tipo, "campo": campo, "campo_presente": campo in texto})

df_campos = pd.DataFrame(chequeos)
if not df_campos.empty:
    resumen_campos = (df_campos.groupby(["tipo", "campo"])["campo_presente"]
                      .agg(unidades_revisadas="size", unidades_con_campo="sum").reset_index())
    resumen_campos["porcentaje_con_campo"] = (resumen_campos["unidades_con_campo"] / resumen_campos["unidades_revisadas"] * 100).round(2)
    display(resumen_campos)
else:
    print("No se pudieron clasificar unidades por nombre. Revisa el inventario y el patrón de nombres.")


### 6.1. Validar fidelidad entre JSON original y unidad textual

Esta comprobación toma una muestra reproducible de unidades, localiza su JSON original mediante la ruta relativa y compara los campos/valores escalares del JSON con el texto generado. Es un control automático de representación, no una prueba semántica completa. Los valores que no se encuentren se muestran como ejemplos para inspección manual. Ajusta `MAX_UNIDADES_FIDELIDAD = None` para revisar todas las unidades; inicialmente se usan hasta 100 para mantener la ejecución rápida.


In [ ]:
# Validación de fidelidad JSON-texto en una muestra reproducible.
# Se comparan claves y valores escalares del JSON con el contenido textual.
import unicodedata

MAX_UNIDADES_FIDELIDAD = 100  # None = revisar todas las unidades


def normalizar_comparacion(valor):
    """Normaliza texto para comparar sin distinguir mayúsculas ni espacios."""
    s = unicodedata.normalize("NFKC", str(valor)).casefold()
    s = re.sub(r"\s+", " ", s).strip()
    return s


def recorrer_escalares(obj, ruta=""):
    """Devuelve (ruta_de_campo, clave, valor) para valores escalares del JSON."""
    encontrados = []
    if isinstance(obj, dict):
        for clave, valor in obj.items():
            ruta_hija = f"{ruta}.{clave}" if ruta else str(clave)
            if isinstance(valor, (dict, list)):
                encontrados.extend(recorrer_escalares(valor, ruta_hija))
            else:
                encontrados.append((ruta_hija, str(clave), valor))
    elif isinstance(obj, list):
        for i, valor in enumerate(obj):
            ruta_hija = f"{ruta}[{i}]"
            if isinstance(valor, (dict, list)):
                encontrados.extend(recorrer_escalares(valor, ruta_hija))
            else:
                encontrados.append((ruta_hija, ruta, valor))
    return encontrados


def localizar_json_para_texto(archivo_txt):
    """Resuelve primero por ruta relativa y sufijo esperado; luego usa metadatos."""
    relativa_txt = archivo_txt.relative_to(RUTA_TEXTOS)
    relativa_json = relativa_txt.with_name(
        relativa_txt.name.removesuffix("_unidad_textual.txt") + ".json"
    )
    candidato = RUTA_JSON / relativa_json
    if candidato.exists():
        return candidato

    # Respaldo para unidades cuya salida conserve un prefijo Peru/.
    partes = list(relativa_json.parts)
    if partes and partes[0].lower() == "peru":
        candidato = RUTA_JSON.joinpath(*partes[1:])
        if candidato.exists():
            return candidato

    # Último recurso: resolver los metadatos ya analizados en la sección 3.
    rel = relativa_txt.as_posix()
    if not df_rastreo.empty:
        fila = df_rastreo.loc[df_rastreo["unidad"] == rel]
        if not fila.empty:
            ruta = fila.iloc[0].get("ruta_origen_resuelta", "")
            if ruta and Path(ruta).exists():
                return Path(ruta)
    return None


unidades_fidelidad = archivos_txt
if MAX_UNIDADES_FIDELIDAD is not None:
    # Muestra reproducible: archivos ordenados por ruta.
    unidades_fidelidad = unidades_fidelidad[:MAX_UNIDADES_FIDELIDAD]

resultados_fidelidad = []
for archivo_txt in unidades_fidelidad:
    ruta_txt = archivo_txt.relative_to(RUTA_TEXTOS).as_posix()
    ruta_json = localizar_json_para_texto(archivo_txt)
    try:
        texto = archivo_txt.read_text(encoding="utf-8-sig")
        if ruta_json is None:
            resultados_fidelidad.append({
                "unidad": ruta_txt, "json_localizado": False,
                "escalares_json": 0, "claves_en_texto_pct": None,
                "valores_en_texto_pct": None, "estado": "JSON no localizado",
                "ejemplos_valores_no_encontrados": ""
            })
            continue

        with ruta_json.open("r", encoding="utf-8-sig") as f:
            objeto = json.load(f)
        escalares = recorrer_escalares(objeto)
        texto_norm = normalizar_comparacion(texto)
        claves = [normalizar_comparacion(clave) for _, clave, _ in escalares if clave]
        claves_presentes = sum(clave in texto_norm for clave in claves)

        valores_comparables = []
        for ruta_campo, clave, valor in escalares:
            # null/None y booleanos pueden representarse de distinta forma; se registran,
            # pero no se usan para el porcentaje de presencia de valores.
            if valor is None or isinstance(valor, bool):
                continue
            valor_norm = normalizar_comparacion(valor)
            if valor_norm:
                valores_comparables.append((ruta_campo, valor_norm))
        valores_faltantes = [ruta for ruta, valor in valores_comparables if valor not in texto_norm]
        pct_claves = (100 * claves_presentes / len(claves)) if claves else None
        pct_valores = (100 * (len(valores_comparables) - len(valores_faltantes)) / len(valores_comparables)) if valores_comparables else None

        resultados_fidelidad.append({
            "unidad": ruta_txt,
            "json_localizado": True,
            "escalares_json": len(escalares),
            "claves_en_texto_pct": round(pct_claves, 2) if pct_claves is not None else None,
            "valores_en_texto_pct": round(pct_valores, 2) if pct_valores is not None else None,
            "estado": "Revisar" if (pct_claves is not None and pct_claves < 100) or (pct_valores is not None and pct_valores < 100) else "Coincidencia textual completa",
            "ejemplos_valores_no_encontrados": "; ".join(valores_faltantes[:5])
        })
    except Exception as exc:
        resultados_fidelidad.append({
            "unidad": ruta_txt, "json_localizado": ruta_json is not None,
            "escalares_json": 0, "claves_en_texto_pct": None,
            "valores_en_texto_pct": None, "estado": f"Error: {str(exc)[:120]}",
            "ejemplos_valores_no_encontrados": ""
        })

df_fidelidad = pd.DataFrame(resultados_fidelidad)
if not df_fidelidad.empty:
    display(pd.DataFrame([
        {"indicador": "Unidades evaluadas", "valor": len(df_fidelidad)},
        {"indicador": "JSON localizados", "valor": int(df_fidelidad["json_localizado"].sum())},
        {"indicador": "Coincidencia textual completa", "valor": int((df_fidelidad["estado"] == "Coincidencia textual completa").sum())},
        {"indicador": "Unidades para revisar", "valor": int((df_fidelidad["estado"] != "Coincidencia textual completa").sum())},
        {"indicador": "Promedio de claves presentes (%)", "valor": round(df_fidelidad["claves_en_texto_pct"].mean(), 2)},
        {"indicador": "Promedio de valores presentes (%)", "valor": round(df_fidelidad["valores_en_texto_pct"].mean(), 2)},
    ]))
    display(df_fidelidad[df_fidelidad["estado"] != "Coincidencia textual completa"].head(30))
    print("Nota: una diferencia puede deberse al formato de serialización y debe inspeccionarse; no implica automáticamente pérdida de datos.")
else:
    print("No se pudo ejecutar la validación: no se encontraron unidades textuales.")


## 7. Revisar contexto geográfico en las unidades

Las rutas de salida deberían conservar la jerarquía de carpetas del JSON original. Esta celda resume la profundidad de las rutas.

In [ ]:
if archivos_txt:
    df_rutas = pd.DataFrame({"ruta": [p.relative_to(RUTA_TEXTOS).as_posix() for p in archivos_txt]})
    df_rutas["profundidad_carpetas"] = df_rutas["ruta"].apply(lambda x: max(0, len(Path(x).parts) - 1))
    display(df_rutas.groupby("profundidad_carpetas").size().rename("cantidad_unidades").reset_index())
    display(df_rutas.head(20))
else:
    print("No hay rutas que revisar.")


## 8. Visualizaciones de la validación

Los gráficos resumen distintos controles de calidad de las unidades textuales: volumen por tipo de archivo, tamaños, trazabilidad hacia el JSON original, conservación de campos, profundidad de carpetas y diferencia entre JSON y textos.

In [ ]:
# 8.1 Cantidad de unidades por tipo de archivo
if not df_textos.empty:
    def clasificar_archivo(nombre):
        n = nombre.lower()
        if "participantes" in n:
            return "Participantes"
        if "totales" in n:
            return "Totales"
        if any(x in n for x in ["departamentos", "provincias", "distritos"]):
            return "Catálogos geográficos"
        return "Otros"

    df_textos["tipo_unidad"] = df_textos["archivo"].apply(clasificar_archivo)
    conteo_tipos = df_textos["tipo_unidad"].value_counts().sort_values(ascending=True)
    plt.figure(figsize=(9, 4.5))
    conteo_tipos.plot(kind="barh")
    plt.title("Unidades textuales por tipo de archivo")
    plt.xlabel("Cantidad de unidades")
    plt.ylabel("")
    for i, v in enumerate(conteo_tipos.values):
        plt.text(v, i, f" {v:,}", va="center")
    plt.tight_layout()
    plt.show()

    # 8.2 Distribución de tamaños: escala logarítmica para distinguir archivos pequeños y grandes.
    tamanos = df_textos.loc[df_textos["tamano_bytes"] > 0, "tamano_bytes"] / 1024
    if not tamanos.empty:
        plt.figure(figsize=(9, 4.5))
        plt.hist(tamanos, bins=30)
        plt.xscale("log")
        plt.title("Distribución del tamaño de las unidades textuales")
        plt.xlabel("Tamaño del archivo (KB, escala logarítmica)")
        plt.ylabel("Cantidad de archivos")
        plt.tight_layout()
        plt.show()

    # 8.3 Calidad de trazabilidad y contenido.
    if not df_rastreo.empty:
        controles = {
            "Metadato de rastreo": df_rastreo["tiene_metadato_rastreo"].mean() * 100,
            "JSON original localizado": df_rastreo["json_origen_existe"].mean() * 100,
            "Contenido no vacío": df_rastreo["tiene_contenido"].mean() * 100,
        }
        serie_controles = pd.Series(controles).sort_values()
        plt.figure(figsize=(9, 4.5))
        serie_controles.plot(kind="barh", xlim=(0, 100))
        plt.title("Porcentaje de unidades que cumplen controles básicos")
        plt.xlabel("Unidades (%)")
        plt.ylabel("")
        for i, v in enumerate(serie_controles.values):
            plt.text(min(v + 1, 96), i, f"{v:.1f}%", va="center")
        plt.tight_layout()
        plt.show()

    # 8.4 Conservación de campos relevantes, por tipo de unidad.
    if not df_campos.empty:
        matriz_campos = resumen_campos.pivot(
            index="tipo", columns="campo", values="porcentaje_con_campo"
        )
        plt.figure(figsize=(12, max(3.5, 0.65 * len(matriz_campos))))
        imagen = plt.imshow(matriz_campos.fillna(0).values, aspect="auto", vmin=0, vmax=100)
        plt.colorbar(imagen, label="Unidades que contienen el campo (%)")
        plt.xticks(range(len(matriz_campos.columns)), matriz_campos.columns, rotation=45, ha="right")
        plt.yticks(range(len(matriz_campos.index)), matriz_campos.index)
        plt.title("Conservación de campos relevantes")
        for i in range(matriz_campos.shape[0]):
            for j in range(matriz_campos.shape[1]):
                valor = matriz_campos.iloc[i, j]
                if pd.notna(valor):
                    plt.text(j, i, f"{valor:.0f}", ha="center", va="center", fontsize=8)
        plt.tight_layout()
        plt.show()

    # 8.5 Distribución de unidades según profundidad de carpetas.
    rutas_graf = pd.DataFrame({
        "ruta": [p.relative_to(RUTA_TEXTOS).as_posix() for p in archivos_txt]
    })
    rutas_graf["profundidad_carpetas"] = rutas_graf["ruta"].apply(
        lambda x: max(0, len(Path(x).parts) - 1)
    )
    profundidad = rutas_graf["profundidad_carpetas"].value_counts().sort_index()
    plt.figure(figsize=(8, 4))
    profundidad.plot(kind="bar")
    plt.title("Unidades textuales por profundidad de ruta")
    plt.xlabel("Cantidad de carpetas en la ruta relativa")
    plt.ylabel("Cantidad de unidades")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

# 8.6 Comparación general del número de JSON y unidades textuales.
if not comparacion.empty:
    conteos_comparacion = comparacion[comparacion["indicador"].isin(
        ["JSON originales", "Unidades textuales"]
    )]
    if not conteos_comparacion.empty:
        plt.figure(figsize=(7, 4))
        plt.bar(conteos_comparacion["indicador"], conteos_comparacion["cantidad"])
        plt.title("Volumen de archivos de origen y unidades generadas")
        plt.ylabel("Cantidad de archivos")
        plt.xticks(rotation=10, ha="right")
        for i, v in enumerate(conteos_comparacion["cantidad"]):
            plt.text(i, v, f" {v:,}", ha="center", va="bottom")
        plt.tight_layout()
        plt.show()
else:
    print("No hay datos suficientes para generar visualizaciones. Revisa las rutas y ejecuta las secciones anteriores.")


## 9. Exportar el diagnóstico

Se guardan tablas CSV dentro de `results/eda_unidades_textuales/` 

In [ ]:
RUTA_RESULTADOS = RAIZ / "results" / "eda_unidades_textuales"
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)
if not df_textos.empty:
    df_textos.to_csv(RUTA_RESULTADOS / "inventario_unidades.csv", index=False, encoding="utf-8-sig")
if not df_rastreo.empty:
    df_rastreo.to_csv(RUTA_RESULTADOS / "validacion_rastreo.csv", index=False, encoding="utf-8-sig")
if not df_campos.empty:
    resumen_campos.to_csv(RUTA_RESULTADOS / "validacion_campos.csv", index=False, encoding="utf-8-sig")
if not comparacion.empty:
    comparacion.to_csv(RUTA_RESULTADOS / "comparacion_json_unidades.csv", index=False, encoding="utf-8-sig")
print("Diagnósticos guardados en:", RUTA_RESULTADOS)
